In [1]:
import sys
import json

# TO CHANGE
BASEDIR = "/home/dzigen/Desktop/PersonalAI/Personal-AI"
# TO CHNAGE

sys.path.insert(0, BASEDIR)

from src.memorize_pipeline import MemPipeline
from src.memorize_pipeline.extractor.LLMExtractor import LLMExtractor
from src.memorize_pipeline.updator.LLMUpdator import LLMUpdator
from src.llm_agent import AgentConnector


DATASET_PATH = '../data/Augment_DiaASQ.json'

In [2]:
with open(DATASET_PATH, 'r', encoding='utf-8') as fd:
    data = json.loads(fd.read())

In [3]:
raw_texts = list(map(lambda v: v['text_dialog'], data['data']))

In [4]:
agent = AgentConnector.open()

In [5]:
agent.generate("Сколько будет 2 + 2?")

'2 + 2 = 4'

In [6]:
extractor = LLMExtractor(agent_conn=agent)

In [39]:
out = extractor.extract_thesises(raw_texts[5])

TEXT: Emily: "My brother 's GT2PRO is not as hot as his IQOO7 calling for a voice call . Xiaomi 10Pro 's video is not hot . I suddenly felt that 10PRO would not get stuck . I hope 9000 will be better ."
Blake: "10Pro 's back cover is broken , still holding on , really strong"

EXTRACTED THESISES: After analyzing the input text, I extracted the following knowledge graph:

"Emily's brother's GT2PRO is not as hot as his IQOO7; ['Emily's brother', 'GT2PRO', 'IQOO7', 'temperature']. Emily's brother's IQOO7 is used for voice call; ['Emily's brother', 'IQOO7', 'voice call']. Xiaomi 10Pro's video is not hot; ['Xiaomi 10Pro', 'video', 'temperature']. Emily suddenly felt that 10PRO would not get stuck; ['Emily', '10PRO','sticking']. Emily hopes 9000 will be better; ['Emily', '9000', 'quality']. 

Blake's 10Pro's back cover is broken; ['Blake', '10Pro', 'back cover', 'broken']. Blake is still holding on; ['Blake', 'holding']. Blake thinks 10Pro is really strong; ['Blake', '10Pro','strength']."

N

In [36]:
raw_triplets = '''Here are the extracted triplets:

Gregory, use, IQOO9; Gregory, feel, fragrant; Gregory, use, IQOO9 for many days; Gregory, feel, very fragrant;

Alan, suggest, listen to digital bloggers; Alan, suggest, buy cost-effective machine; Alan, ask, does everyone need it; Alan, ask, does everyone play Yuanshen;

Kyle, ask, is there still a cost-effective machine; Kyle, ask, does hardware shrink; Kyle, ask, does price rise; Kyle, laugh, haha;

Alan, state, sub-brand is cost-effective; Alan, state, hardware shrinks; Alan, state, price rises;

Deborah, state, IQOO9 is fragrant; Deborah, suggest, try again in summer;

Gregory, state, IQOO9 is fever; Gregory, state, IQOO9 is hot; Gregory, state, IQOO9 is sweaty;

Ann, state, game experience is stabilized; Ann, state, frame rate is high; Ann, state, TM's 888 and 8gen1 is hot; Ann, state, TM's 888 and 8gen1 is slow;

Amanda, state, IQOO9 scheduling is conservative; Amanda, state, Xiaomi Mi 12Pro is unhot; Amanda, state, manufacturers are conservative; Amanda, state, large nuclear frequency is low;

Gregory, state, IQOO9's heat dissipation stack is not good; Gregory, state, Xiaomi Mi 12Pro's heat dissipation stack is not good;

Jordan, state, hot plate area is almost the same;

Note: I've extracted the triplets as per the guidelines provided, breaking down complex sentences into simpler ones and avoiding assumptions. Let me know if you'd like me to clarify or modify any of the extracted triplets!'''

In [15]:
node_prop, rel_prop = {}, {}

In [31]:
' '.join(list(filter(lambda v: len(v), raw_triplets.split("\n")[1:-1])))

"Gregory, use, IQOO9; Gregory, feel, fragrant; Gregory, use, IQOO9 for many days; Gregory, feel, very fragrant; Alan, suggest, listen to digital bloggers; Alan, suggest, buy cost-effective machine; Alan, ask, does everyone need it; Alan, ask, does everyone play Yuanshen; Kyle, ask, is there still a cost-effective machine; Kyle, ask, does hardware shrink; Kyle, ask, does price rise; Kyle, laugh, haha; Alan, state, sub-brand is cost-effective; Alan, state, hardware shrinks; Alan, state, price rises; Deborah, state, IQOO9 is fragrant; Deborah, suggest, try again in summer; Gregory, state, IQOO9 is fever; Gregory, state, IQOO9 is hot; Gregory, state, IQOO9 is sweaty; Ann, state, game experience is stabilized; Ann, state, frame rate is high; Ann, state, TM's 888 and 8gen1 is hot; Ann, state, TM's 888 and 8gen1 is slow; Amanda, state, IQOO9 scheduling is conservative; Amanda, state, Xiaomi Mi 12Pro is unhot; Amanda, state, manufacturers are conservative; Amanda, state, large nuclear frequenc

In [37]:
raw_triplets = ' '.join(list(filter(lambda v: len(v), raw_triplets.split("\n")[1:-1]))).lower()
raw_triplets = raw_triplets.split(";")
triplets = []
for triplet in raw_triplets:
    if len(triplet.split(",")) != 3:
        continue
    subj, rel, obj = triplet.split(",")
    subj, rel, obj = subj.split(":")[-1].split(".")[-1].strip(''' \n'".,/'''), rel.strip(''' \n'".,/'''), obj.strip(''' \n'".,/''')
    if len(subj) == 0 or len(rel) == 0 or len(obj) == 0:
        continue
    triplets.append(
        [
            {"name": subj, "type": "object", "prop": {**node_prop}},
            {"name": rel, "prop": {"type": "simple", **rel_prop}},
            {"name": obj, "type": "object", "prop": {**node_prop}}
        ]
    )

In [38]:
triplets

[[{'name': 'gregory', 'type': 'object', 'prop': {}},
  {'name': 'use', 'prop': {'type': 'simple'}},
  {'name': 'iqoo9', 'type': 'object', 'prop': {}}],
 [{'name': 'gregory', 'type': 'object', 'prop': {}},
  {'name': 'feel', 'prop': {'type': 'simple'}},
  {'name': 'fragrant', 'type': 'object', 'prop': {}}],
 [{'name': 'gregory', 'type': 'object', 'prop': {}},
  {'name': 'use', 'prop': {'type': 'simple'}},
  {'name': 'iqoo9 for many days', 'type': 'object', 'prop': {}}],
 [{'name': 'gregory', 'type': 'object', 'prop': {}},
  {'name': 'feel', 'prop': {'type': 'simple'}},
  {'name': 'very fragrant', 'type': 'object', 'prop': {}}],
 [{'name': 'alan', 'type': 'object', 'prop': {}},
  {'name': 'suggest', 'prop': {'type': 'simple'}},
  {'name': 'listen to digital bloggers', 'type': 'object', 'prop': {}}],
 [{'name': 'alan', 'type': 'object', 'prop': {}},
  {'name': 'suggest', 'prop': {'type': 'simple'}},
  {'name': 'buy cost-effective machine', 'type': 'object', 'prop': {}}],
 [{'name': 'alan'

In [20]:
triplet.split(",")

[" i've extracted the triplets as per the guidelines provided",
 " breaking down complex sentences into simpler ones and avoiding assumptions. let me know if you'd like me to clarify or modify any of the extracted triplets!"]

In [ ]:
# Проверить работоспособность каждого класса в отдельности 

# попробовать создать новую бд

# добавить в бд новый граф